<a href="https://colab.research.google.com/github/DiploDatos/AnalisisYCuracion/blob/master/Entregable_Parte_2_2022.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Diplomatura en Ciencia de Datos, Aprendizaje Autom?tico y sus Aplicaciones**

**Exploraci?n y Curaci?n de Datos**

*Edici?n 2021*

----

# Trabajo pr?ctico entregable - parte 2


En el ejercicio 2 de la parte 1 del entregable seleccionaron las filas y columnas relevantes al problema de predicci?n de precios de una propiedad e imputaron los valores faltantes de las columnas `Suburb` y las columnas obtenidas a partir del conjunto de datos `airbnb`.

En esta notebook, **se utilizar? resultado de dichas operaciones.** que fue guardado en un archivo (Ejercicio 3 entregable 1). 


In [ ]:
import matplotlib.pyplot as plt
import numpy
import pandas

import seaborn
seaborn.set_context('talk')

In [ ]:
# Ac? deber?an leer el conjunto de datos que ya tienen.
melb_df = pandas.read_csv(
    'https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/melb_data.csv')
melb_df[:3]

## Ejercicio 1: Encoding

1. Seleccionar todas las filas y columnas del conjunto de datos obtenido en la parte 1 del entregable, **excepto** `BuildingArea` y `YearBuilt`, que volveremos a imputar m?s adelante.

2. Aplicar una codificaci?n One-hot encoding a cada fila, tanto para variables num?ricas como categ?ricas. Si lo consideran necesario, pueden volver a reducir el n?mero de categor?as ?nicas.

Algunas opciones:
  1. Utilizar `OneHotEncoder` junto con el par?metro `categories` para las variables categ?ricas y luego usar `numpy.hstack` para concatenar el resultado con las variables num?ricas. 
  2. `DictVectorizer` con algunos pasos de pre-proceso previo.

Recordar tambi?n que el atributo `pandas.DataFrame.values` permite acceder a la matriz de numpy subyacente a un DataFrame.


In [ ]:
categorical_cols = ['Type']
numerical_cols = ['Rooms']

In [ ]:
melb_df[categorical_cols].nunique()

In [ ]:
# Check for nulls
melb_df[categorical_cols].isna().sum()

## Ejercicio 2: Imputaci?n por KNN

En el te?rico se present? el m?todo `IterativeImputer` para imputar valores faltantes en variables num?ricas. Sin embargo, los ejemplos presentados s?lo utilizaban algunas variables num?ricas presentes en el conjunto de datos. En este ejercicio, utilizaremos la matriz de datos codificada para imputar datos faltantes de manera m?s precisa.

1. Agregue a la matriz obtenida en el punto anterior las columnas `YearBuilt` y `BuildingArea`.
2. Aplique una instancia de `IterativeImputer` con un estimador `KNeighborsRegressor` para imputar los valores de las variables. ?Es necesario estandarizar o escalar los datos previamente?
3. Realice un gr?fico mostrando la distribuci?n de cada variable antes de ser imputada, y con ambos m?todos de imputaci?n.

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.neighbors import KNeighborsRegressor
from sklearn.impute import IterativeImputer

melb_data_mice = melb_df.copy(deep=True)

mice_imputer = IterativeImputer(random_state=0, estimator=KNeighborsRegressor())
melb_data_mice[['YearBuilt','BuildingArea']] = mice_imputer.fit_transform(
    melb_data_mice[['YearBuilt', 'BuildingArea']])

Ejemplo de gr?fico comparando las distribuciones de datos obtenidas con cada m?todo de imputaci?n.

In [ ]:
mice_year_built = melb_data_mice.YearBuilt.to_frame()
mice_year_built['Imputation'] = 'KNN over YearBuilt and BuildingArea'
melb_year_build = melb_df.YearBuilt.dropna().to_frame()
melb_year_build['Imputation'] = 'Original'
data = pandas.concat([mice_year_built, melb_year_build])
fig = plt.figure(figsize=(8, 5))
g = seaborn.kdeplot(data=data, x='YearBuilt', hue='Imputation')

## Ejercicio 3: Reducci?n de dimensionalidad.

Utilizando la matriz obtenida en el ejercicio anterior:
1. Aplique `PCA` para obtener $n$ componentes principales de la matriz, donde `n = min(20, X.shape[0])`. ?Es necesario estandarizar o escalar los datos?
2. Grafique la varianza capturada por los primeros $n$ componentes principales, para cada $n$.
3. En base al gr?fico, seleccione las primeras $m$ columnas de la matriz transformada para agregar como nuevas caracter?sticas al conjunto de datos.

## Ejercicio 4: Composici?n del resultado

Transformar nuevamente el conjunto de datos procesado en un `pandas.DataFrame` y guardarlo en un archivo.

Para eso, ser? necesario recordar el nombre original de cada columna de la matriz, en el orden correcto. Tener en cuenta:
1. El m?todo `OneHotEncoder.get_feature_names` o el atributo `OneHotEncoder.categories_` permiten obtener una lista con los valores de la categor?a que le corresponde a cada ?ndice de la matriz.
2. Ninguno de los m?todos aplicados intercambia de lugar las columnas o las filas de la matriz.

In [ ]:
## Small example
from sklearn.decomposition import PCA
from sklearn.preprocessing import OneHotEncoder

## If we process our data with the following steps:
categorical_cols = ['Type', 'Regionname']
numerical_cols = ['Rooms', 'Distance']
new_columns = []

# Step 1: encode categorical columns
encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
X_cat = encoder.fit_transform(melb_df[categorical_cols])
for col, col_values in zip(categorical_cols, encoder.categories_):
  for col_value in col_values:
    new_columns.append('{}={}'.format(col, col_value))
print("Matrix has shape {}, with columns: {}".format(X_cat.shape, new_columns))

# Step 2: Append the numerical columns
X = numpy.hstack([X_cat, melb_df[numerical_cols].values])
new_columns.extend(numerical_cols)
print("Matrix has shape {}, with columns: {}".format(X_cat.shape, new_columns))

# Step 3: Append some new features, like PCA
pca = PCA(n_components=2)
pca_dummy_features = pca.fit_transform(X)
X_pca = numpy.hstack([X, pca_dummy_features])
new_columns.extend(['pca1', 'pca2'])

## Re-build dataframe
processed_melb_df = pandas.DataFrame(data=X_pca, columns=new_columns)
processed_melb_df.head()

## Ejercicio 5: Documentaci?n

En un documento `.pdf` o `.md` realizar un reporte de las operaciones que realizaron para obtener el conjunto de datos final. Se debe incluir:
  1. Criterios de exclusi?n (o inclusi?n) de filas
  2. Interpretaci?n de las columnas presentes
  2. Todas las transofrmaciones realizadas

Este documento es de uso t?cnico exclusivamente, y su objetivo es permitir que otres desarrolladores puedan reproducir los mismos pasos y obtener el mismo resultado. Debe ser detallado pero consiso. Por ejemplo:

```
  ## Criterios de exclusi?n de ejemplos
  1. Se eliminan ejemplos donde el a?o de construcci?n es previo a 1900

  ## Caracter?sticas seleccionadas
  ### Caracter?sticas categ?ricas
  1. Type: tipo de propiedad. 3 valores posibles
  2. ...
  Todas las caracter?sticas categ?ricas fueron codificadas con un
  m?todo OneHotEncoding utilizando como m?ximo sus 30 valores m?s 
  frecuentes.
  
  ### Caracter?sticas num?ricas
  1. Rooms: Cantidad de habitaciones
  2. Distance: Distancia al centro de la ciudad.
  3. airbnb_mean_price: Se agrega el precio promedio diario de 
     publicaciones de la plataforma AirBnB en el mismo c?digo 
     postal. [Link al repositorio con datos externos].

  ### Transformaciones:
  1. Todas las caracter?sticas num?ricas fueron estandarizadas.
  2. La columna `Suburb` fue imputada utilizando el m?todo ...
  3. Las columnas `YearBuilt` y ... fueron imputadas utilizando el 
     algoritmo ...
  4. ...

  ### Datos aumentados
  1. Se agregan las 5 primeras columnas obtenidas a trav?s del
     m?todo de PCA, aplicado sobre el conjunto de datos
     totalmente procesado.
```
